# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List record set @ids, their names, and field @ids
record_sets = list(dataset.record_sets)

print('Available record sets:')
for rset in record_sets:
    print(f"Record set @id: {rset.id} | name: {getattr(rset, 'name', 'N/A')}")
    if hasattr(rset, 'fields'):
        print('  Available fields:')
        for field in rset.fields:
            print(f"    Field @id: {field.id}, name: {getattr(field, 'name', 'N/A')}, dataType: {getattr(field, 'data_type', 'N/A')}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Select record set(s) to extract as DataFrame
record_set_ids = [rset.id for rset in dataset.record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    print(f"\nLoading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        dataframes[record_set_id] = pd.DataFrame(records)
        print("Loaded columns:", dataframes[record_set_id].columns.tolist())
        print(dataframes[record_set_id].head(2))
    else:
        print("No records found.")

# For demonstration, select the first record set for further analysis
main_record_set_id = record_set_ids[0]
df = dataframes[main_record_set_id]

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Identify a numeric field for analysis by scanning the DataFrame for numeric columns.
numeric_candidates = df.select_dtypes(include=['number']).columns.tolist()
print("Numeric candidate fields:", numeric_candidates)

# Pick the first numeric field for demonstration (can be adjusted to a meaningful field after inspection)
if numeric_candidates:
    numeric_field = numeric_candidates[0]
    threshold = df[numeric_field].mean()  # Use mean as a demo threshold
    filtered_df = df[df[numeric_field] > threshold].copy()

    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Attempt to group by a categorical field (select the first non-numeric column)
    group_candidates = [c for c in df.columns if c not in numeric_candidates]
    if group_candidates:
        group_field = group_candidates[0]
        print(f"\nGrouping by field: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(grouped_df.head())
else:
    print('No numeric fields found for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'numeric_field' in locals() and numeric_candidates:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Histogram of the numeric field
    sns.histplot(df[numeric_field].dropna(), ax=axes[0], kde=True)
    axes[0].set_title(f"Distribution of {numeric_field}")

    # Boxplot of the numeric field grouped by the group field (if available)
    if 'group_field' in locals():
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df, ax=axes[1])
        axes[1].set_title(f"{numeric_field} by {group_field}")
        axes[1].tick_params(axis='x', rotation=45)
    else:
        # Otherwise, just show scatter
        axes[1].scatter(filtered_df.index, filtered_df[numeric_field])
        axes[1].set_title(f"{numeric_field} scatter plot")
    plt.tight_layout()
    plt.show()
else:
    print('No numeric field found for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we:
- Loaded a Croissant-annotated mass spectrometry dataset using `mlcroissant`.
- Explored available record sets and their fields using their `@id` for consistency.
- Extracted data using these references and examined the structure in pandas DataFrames.
- Performed a basic EDA: filtered records by a numeric field, normalized its values, and grouped the data for summarized insights.
- Visualized data distributions to understand patterns.

This workflow demonstrates step-by-step data exploration using Croissant metadata and reproducible, schema-driven methods. For deeper analyses, review domain-specific fields and tailor EDA and ML to your research questions.